In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


In [2]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [4]:
# One-hot encode Geography & Gender (keep all categories)
df_encoded = pd.get_dummies(
    df,
    columns=["Geography", "Gender"],
    drop_first=False
).drop(columns=["RowNumber", "CustomerId", "Surname"])

df_encoded.head()


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,True,False,False,True,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,False,True,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,True,False,False,True,False
3,699,39,1,0.00,2,0,0,93826.63,0,True,False,False,True,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,False,True,True,False


In [5]:
#LOGISTIC REGRESSION

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Split X / y
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

#Train/test split (stratified for class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Scale only continuous numeric columns; pass everything else through
numeric_scaled = ["CreditScore","Age","Tenure","Balance","NumOfProducts","EstimatedSalary"]

preprocess = ColumnTransformer(
    transformers=[("scale", StandardScaler(), numeric_scaled)],
    remainder="passthrough"
)

# Pipeline: preprocessing + Logistic Regression
clf = Pipeline([
    ("prep", preprocess),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

clf.fit(X_train, y_train)

# Evaluate
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, proba))
print("\nClassification report:\n", classification_report(y_test, pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))



Accuracy: 0.7135
ROC-AUC : 0.7771762517525228

Classification report:
               precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.70      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000


Confusion matrix:
 [[1142  451]
 [ 122  285]]


In [6]:
#RANDOM FOREST

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

#  Features & target
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Random Forest (good defaults)
rf = RandomForestClassifier(
    n_estimators=300,        
    max_features="sqrt",    
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Evaluate
pred  = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, proba))
print("\nClassification report:\n", classification_report(y_test, pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))


Accuracy: 0.859
ROC-AUC : 0.8533680059103786

Classification report:
               precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.77      0.44      0.56       407

    accuracy                           0.86      2000
   macro avg       0.82      0.70      0.74      2000
weighted avg       0.85      0.86      0.84      2000


Confusion matrix:
 [[1538   55]
 [ 227  180]]


In [8]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Features & target
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Imbalance handling
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

# XGBoost model
gb = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

# Fit 
gb.fit(X_train, y_train)

#  Evaluate
pred  = gb.predict(X_test)
proba = gb.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, proba))
print("\nClassification report:\n", classification_report(y_test, pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))


Accuracy: 0.8085
ROC-AUC : 0.8589915030592996

Classification report:
               precision    recall  f1-score   support

           0       0.92      0.83      0.87      1593
           1       0.52      0.72      0.60       407

    accuracy                           0.81      2000
   macro avg       0.72      0.77      0.74      2000
weighted avg       0.84      0.81      0.82      2000


Confusion matrix:
 [[1325  268]
 [ 115  292]]


In [10]:
# CatBoost (Gradient Boosting) 
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

#  Features & target
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Model (good defaults; handles imbalance with auto weights)
cb = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False
)

# Fit
cb.fit(X_train, y_train)

# Evaluate
pred  = cb.predict(X_test)
proba = cb.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, proba))
print("\nClassification report:\n", classification_report(y_test, pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))


Accuracy: 0.823
ROC-AUC : 0.847142982736203

Classification report:
               precision    recall  f1-score   support

           0       0.91      0.86      0.89      1593
           1       0.55      0.67      0.61       407

    accuracy                           0.82      2000
   macro avg       0.73      0.77      0.75      2000
weighted avg       0.84      0.82      0.83      2000


Confusion matrix:
 [[1373  220]
 [ 134  273]]


In [12]:
# LightGBM (Gradient Boosting)
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Features & target
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Model (handles imbalance with class weights)
lgbm = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",   # helps with churn imbalance
    random_state=42,
    n_jobs=-1
)

# Fit
lgbm.fit(X_train, y_train)

# Evaluate
pred  = lgbm.predict(X_test)
proba = lgbm.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, proba))
print("\nClassification report:\n", classification_report(y_test, pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))


[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000949 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 861
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Accuracy: 0.8335
ROC-AUC : 0.8507783592529354

Classification report:
               precision    recall  f1-score   support

           0       0.91      0.88      0.89      1593
           1       0.58      0.65      0.62       407

    accuracy                           0.83      2000
   macro avg       0.74      0.77      0.75      2000
weighted avg       0.84      0.83      0.84      2000


Confusion matrix:
 [[1401  192]
 [ 141  266]]


In [13]:
import pandas as pd

# Build prediction table for the Test Set
results = pd.DataFrame({
    "CustomerId": df.loc[X_test.index, "CustomerId"],
    "Surname": df.loc[X_test.index, "Surname"],
    "Exited_actual": y_test.values,
    "Exited_pred": pred,
    "Churn_Prob": proba
})

# Sort by predicted churn risk (highest first)
results = results.sort_values("Churn_Prob", ascending=False)

# Look at the top 20 highest-risk customers
print(results.head(20))


      CustomerId      Surname  Exited_actual  Exited_pred  Churn_Prob
5950    15806808         Hope              1            1    0.999943
9540    15634551   Williamson              1            1    0.999906
6831    15696989  Chukwueloka              1            1    0.999872
3549    15647725   Napolitano              1            1    0.999803
8396    15655082         Pape              1            1    0.999799
555     15775318           Lu              1            1    0.999731
6255    15589017         Chiu              1            1    0.999621
5010    15719508        Davis              1            1    0.999407
70      15703793   Konovalova              1            1    0.999241
871     15692750     McGregor              1            1    0.999232
5922    15786196          Han              1            1    0.999231
1701    15605279      Francis              1            1    0.999157
7435    15647898      Russell              1            1    0.999154
5363    15663410    